# Task 1.1: Core Contribution / Architecture

## Paper: "Kernel Methods for Deep Learning" — Cho & Saul (NIPS 2009)

### Step-by-Step Method Description

**Step 1: Compute the angle between input vectors**

- Given two input vectors **x** and **y** in ℝ^d, we first compute the angle θ between them using the standard formula: θ = cos⁻¹( (x · y) / (‖x‖ ‖y‖) ).
- **Reference:** Equation 2, Section 2.1.
- **Purpose:** The entire arc-cosine kernel family depends on this angle. Unlike RBF kernels that care about Euclidean distance, arc-cosine kernels decompose similarity into an angular part and a magnitude part, which turns out to mirror how threshold networks process their inputs.

**Step 2: Evaluate the arc-cosine kernel of degree n (single-layer kernel)**

- Using the angle θ from Step 1, compute the kernel value as: k_n(x, y) = (1/π) · ‖x‖ⁿ · ‖y‖ⁿ · Jn(θ), where Jn(θ) is a closed-form function that depends on the degree n. For n=0 (step activation), J₀(θ) = π − θ. For n=1 (ramp/ReLU activation), J₁(θ) = sin θ + (π − θ) cos θ. For n=2 (quarter-pipe activation), J₂(θ) = 3 sin θ cos θ + (π − θ)(1 + 2 cos²θ).
- **Reference:** Equations 3–7, Section 2.1; Figure 1 for the activation functions.
- **Purpose:** This is the paper's first core contribution — a family of kernels that each correspond to the inner product in the feature space of an *infinite-width single-layer threshold network* with a specific activation function. Essentially, instead of training a massive neural net, you just evaluate this kernel formula. The degree n controls which activation function the kernel mimics (step, ramp, quarter-pipe). This is derived from the integral representation in Eq. 1, where taking weights as i.i.d. Gaussians in an infinite-width network recovers these kernels exactly (Eq. 8 → Eq. 1 connection).

**Step 3: Compose kernels recursively to form multilayer arc-cosine kernels**

- To simulate a *multilayer* neural net, the authors compose the single-layer kernel from Step 2 with itself recursively. For a kernel of depth ℓ+1, we use: k_n^(ℓ+1)(x, y) = (1/π) · [k_n^(ℓ)(x,x) · k_n^(ℓ)(y,y)]^(n/2) · Jn(θ_n^(ℓ)), where θ_n^(ℓ) = cos⁻¹( k_n^(ℓ)(x,y) / √[k_n^(ℓ)(x,x) · k_n^(ℓ)(y,y)] ). The base case (ℓ=1) is just the single-layer kernel from Step 2.
- **Reference:** Equations 12 and 13, Section 2.3.
- **Purpose:** This recursive composition is *the key theoretical insight*. Unlike polynomial or RBF kernels, whose compositions are trivial or uninteresting (Eqs. 10–11), composing arc-cosine kernels genuinely produces new, richer feature representations at each level, mimicking the hierarchical processing of deep neural networks. The authors discovered empirically that n=1 kernels must be used at layers ℓ > 1 because only n=1 preserves input norms (k₁(x,x) = ‖x‖²), while n=0 collapses everything to a unit sphere and n>1 blows up dynamic range.

**Step 4: Use the (possibly multilayer) kernel in an SVM for classification**

- Once you have the kernel matrix (either single-layer or multilayer), you plug it into a standard SVM (the authors used libSVM). The margin penalty parameter C is tuned via cross-validation on a held-out portion of the training set.
- **Reference:** Section 2.4; experimental methodology follows [11].
- **Purpose:** This is the shallow architecture path — even though the kernel internally encodes deep feature transformations, the classifier itself is still an SVM. This lets you get benefits of depth without the painful non-convex optimization of deep belief nets.

**Step 5 (MKM path): Layer-wise kernel PCA for deep representation learning**

- For the Multilayer Kernel Machine (MKM) architecture, instead of plugging the multilayer kernel directly into an SVM, the authors train a deep architecture layer by layer. At each layer: (a) compute kernel PCA using an arc-cosine kernel to project training data into a nonlinear feature space (extracting top principal components), then (b) pass these components as inputs to the next layer.
- **Reference:** Section 3.1, first subsection on "Kernel PCA"; connection to [13] and [16].
- **Purpose:** This is the unsupervised deep learning step, analogous to greedy layer-wise pre-training in deep belief nets. Each application of kernel PCA extracts increasingly abstract, distributed features from the data.

**Step 6 (MKM path): Feature selection via mutual information at each layer**

- At every layer (including the raw input layer), the MKM ranks features by their estimated mutual information with the class labels. Features below a threshold are pruned. The threshold (number of retained features w) and the k for kNN evaluation are determined by cross-validation on a held-out set.
- **Reference:** Section 3.1, subsection on "Feature selection"; methodology from [14].
- **Purpose:** Without this step, kernel PCA would waste representational capacity on directions that carry no discriminative signal. This supervised pruning focuses the unsupervised learning on the parts of the input that actually matter for classification. It also determines the effective "width" of each layer in the MKM architecture (e.g., 300-90-105-136-126-240 for mnist-back-rand).

**Step 7 (MKM path): Large Margin Nearest Neighbor (LMNN) classification at the output layer**

- After all layers of kernel PCA + feature selection are complete, the final feature representations are fed into an LMNN classifier, which learns a Mahalanobis distance metric optimized for nearest-neighbor classification via semidefinite programming. Test examples are then classified using the energy-based decision rule from [15].
- **Reference:** Section 3.1, subsection on "Distance metric learning"; [15] and [19].
- **Purpose:** LMNN acts as the supervised fine-tuning step for MKMs, similar to how backpropagation fine-tunes all layers of a deep belief net after unsupervised pre-training. The advantage is that LMNN's optimization is convex, avoiding the messy non-convex landscape of neural net training.

---

### Final Summary Sentence

This paper addresses the problem of bridging deep learning and kernel methods by introducing **arc-cosine kernels** — a new kernel family whose feature spaces exactly correspond to infinite-width threshold networks — and showing that these kernels can be recursively composed (for SVMs) or stacked via kernel PCA (for MKMs) to capture the benefits of deep architectures. The authors claim their approach is superior to existing alternatives because it avoids the non-convex optimization headaches of deep belief nets and the need to tune continuous hyperparameters like RBF kernel widths, while still achieving competitive or better classification accuracies on challenging benchmarks.